# Translation to QNAasm

In this notebook, we will see how we can translate a MimiQ circuit into a QNAasm circuit.

## Translating a General Circuit

Firstly, we can need a MimiQ circuit to play with.

In [1]:
from mimiqcircuits import (
    Circuit,
    GateH,
    GateCX,
    Measure,
    IfStatement,
    BitString,
)

c = Circuit()
c.push(GateH(), 0)
c.push(GateCX(), 0, 1)
c.push(GateCX(), 0, 2)
c.push(Measure(), [0, 1, 2], [0, 1, 2])
c.push(GateH(), 3)
c.push(Measure(), 3, 3)
c.push(IfStatement(GateCX(), BitString("01")), *[1, 2], *[3, 1])

c.draw()

        ┌─┐      ┌──────┐                                                       
 q[0]: ╶┤H├─●──●─┤  M   ├──────────────────────────────────────────────────────╴
        └─┘┌┴┐ │ └───╥──┘┌──────┐                               ┌────┐          
 q[1]: ╶───┤X├─┼─────╫───┤  M   ├───────────────────────────────┤0   ├─────────╴
           └─┘┌┴┐    ║   └───╥──┘┌──────┐                       │  CX│          
 q[2]: ╶──────┤X├────╫───────╫───┤  M   ├───────────────────────┤1   ├─────────╴
              └─┘    ║       ║   └───╥──┘┌─┐┌──────┐            └──╥─┘          
 q[3]: ╶─────────────╫───────╫───────╫───┤H├┤  M   ├───────────────╫───────────╴
                     ║       ║       ║   └─┘└───╥──┘               ║            
                     ║       ║       ║          ║                  ║            
                     ║       ║       ║          ║   ┌──────────┐○══╝            
 c:    ══════════════╩═══════╩═══════╩══════════╩═══╪c[3,1]==01╪════════════════
                     0      

Then, we can map each qubit to a position on the 2D grid of the QPU.

In [2]:
from qnaasm.position import Position

qubits = {
    0: Position(0, 0),
    1: Position(4, 0),
    2: Position(4, 4),
    3: Position(2, 0),
}

Translation to QNAasm is straightforward with the `translate` function.

In [3]:
from translate.translation import translate

instructions = translate(c, qubits)

Finally, we can see the translated code by using the pretty printer.

In [4]:
from qnaasm.pretty_printer import PrettyPrinter

printer = PrettyPrinter()

for instr in instructions:
    instr.accept(printer)

calloc c[4]
calloc drop[1]
qalloc {0,0}
qalloc {4,0}
qalloc {4,4}
qalloc {2,0}
gate H {0,0}
move {0,0} to {1,0}
move {2,0} to {2,1}
move {1,0} to {2,0}
move {2,0} to {3,0}
gate CX {3,0},{4,0}
move {4,0} to {4,-1}
move {3,0} to {4,0}
move {4,0} to {4,1}
move {4,1} to {4,2}
move {4,2} to {4,3}
gate CX {4,3},{4,4}
move {4,3} to {4,2}
move {4,2} to {4,1}
move {4,1} to {4,0}
move {4,0} to {3,0}
move {4,-1} to {4,0}
move {3,0} to {2,0}
move {2,0} to {1,0}
move {2,1} to {2,0}
move {1,0} to {0,0}
measure {0,0} to c[0]
measure {4,0} to c[1]
measure {4,4} to c[2]
gate H {2,0}
measure {2,0} to c[3]
if c[1] = 1 then
  if c[3] = 0 then
    {
      move {4,0} to {4,1}
      move {4,1} to {4,2}
      move {4,2} to {4,3}
      gate CX {4,3},{4,4}
      move {4,3} to {4,2}
      move {4,2} to {4,1}
      move {4,1} to {4,0}
    }
qfree {0,0}
qfree {4,0}
qfree {4,4}
qfree {2,0}


## Translating Corrected Circuit With Surface Codes

Now that we can translate a general circuit to QNAasm, it might be interesting to translate a corrected circuit with surface codes. We can do this by calling the function `create_all_surface_code_mapping_for_circuit` to map each qubit of each surface code on the 2D grid.

Then, we can call `translate` as before.

Load a corrected circuit.

In [5]:
c = Circuit.loadproto(f"../circuits/corrected_test.pb")

c.draw()

                                                                         ┌─┐    
 q[0]:  ╶────────────────────────────────────────────────────────────────┤X├───╴
                                                                         └┬┘    
 q[1]:  ╶─────────────────────────────────────────────────────────────────┼────╴
                                                                          │     
 q[2]:  ╶─────────────────────────────────────────────────────────────────┼────╴
                                                                          │     
 q[3]:  ╶─────────────────────────────────────────────────────────────────┼────╴
                                                                          │     
 q[4]:  ╶─────────────────────────────────────────────────────────────────┼────╴
                                                                          │     
 q[5]:  ╶─────────────────────────────────────────────────────────────────┼────╴
                            

Place each qubit on the 2D grid.

In [6]:
from translate.surface_code_mapping import create_all_surface_code_mapping_for_circuit

qubits = create_all_surface_code_mapping_for_circuit(c)

In [7]:
print("Surface Code qubits mapping.")
for qubit, pos in qubits.items():
    print(f"{qubit}: {pos}")

Surface Code qubits mapping.
0: {6,6}
1: {4,6}
2: {2,6}
3: {6,8}
4: {4,8}
5: {2,8}
6: {6,10}
7: {4,10}
8: {2,10}
9: {5,7}
10: {3,9}
11: {3,5}
12: {5,11}
13: {3,7}
14: {5,9}
15: {7,7}
16: {1,9}
17: {14,6}
18: {12,6}
19: {10,6}
20: {14,8}
21: {12,8}
22: {10,8}
23: {14,10}
24: {12,10}
25: {10,10}
26: {13,7}
27: {11,9}
28: {11,5}
29: {13,11}
30: {11,7}
31: {13,9}
32: {15,7}
33: {9,9}


Translate the circuit in QNAasm.

In [8]:
instructions = translate(c, qubits)

Print it.

In [9]:
printer = PrettyPrinter()

for instr in instructions:
    instr.accept(printer)

calloc c[18]
calloc drop[1]
qalloc {6,6}
qalloc {4,6}
qalloc {2,6}
qalloc {6,8}
qalloc {4,8}
qalloc {2,8}
qalloc {6,10}
qalloc {4,10}
qalloc {2,10}
qalloc {5,7}
qalloc {3,9}
qalloc {3,5}
qalloc {5,11}
qalloc {3,7}
qalloc {5,9}
qalloc {7,7}
qalloc {1,9}
qalloc {14,6}
qalloc {12,6}
qalloc {10,6}
qalloc {14,8}
qalloc {12,8}
qalloc {10,8}
qalloc {14,10}
qalloc {12,10}
qalloc {10,10}
qalloc {13,7}
qalloc {11,9}
qalloc {11,5}
qalloc {13,11}
qalloc {11,7}
qalloc {13,9}
qalloc {15,7}
qalloc {9,9}
measure {5,7} to drop
qfree {5,7}
qalloc {5,7}
measure {3,9} to drop
qfree {3,9}
qalloc {3,9}
measure {3,5} to drop
qfree {3,5}
qalloc {3,5}
measure {5,11} to drop
qfree {5,11}
qalloc {5,11}
measure {3,7} to drop
qfree {3,7}
qalloc {3,7}
measure {5,9} to drop
qfree {5,9}
qalloc {5,9}
measure {7,7} to drop
qfree {7,7}
qalloc {7,7}
measure {1,9} to drop
qfree {1,9}
qalloc {1,9}
gate H {5,7}
gate H {3,9}
gate H {3,5}
gate H {5,11}
gate H {3,7}
gate H {5,9}
gate H {7,7}
gate H {1,9}
move {5,7} to {6,7}
ga